# Lab 02: Hybrid RAG — BM25 + Dense Retrieval with RRF

**Goal:** Build a hybrid retrieval pipeline that combines keyword search (BM25) with semantic search (dense embeddings), merges results using Reciprocal Rank Fusion (RRF), and generates answers via the configured LLM provider (`AI_PROVIDER` in `.env`).

**What you'll build:**
1. BM25 index with `rank_bm25`
2. Dense index with `sentence-transformers` + FAISS
3. RRF merge function
4. End-to-end hybrid RAG pipeline
5. Comparison: BM25 only vs. dense only vs. hybrid

**When to use hybrid:** Corpora with product codes, technical terms, or proper nouns that dense embeddings handle poorly (e.g., `SKU-4821-B` or `BERT-base-uncased`).

In [ ]:
!pip install rank-bm25 sentence-transformers faiss-cpu google-genai anthropic openai python-dotenv --quiet

## 1. Sample Corpus

A small corpus that mixes semantic content with exact-match keywords — the ideal test case for hybrid retrieval.

In [ ]:
CORPUS = [
    {"id": "doc_0", "text": "HNSW (Hierarchical Navigable Small World) is the default indexing algorithm for most vector databases. It builds a multi-layer graph for approximate nearest neighbor search in O(log N) time."},
    {"id": "doc_1", "text": "IVF (Inverted File Index) partitions vectors into Voronoi cells. At query time, only nearby cells are searched, reducing cost for billion-scale corpora."},
    {"id": "doc_2", "text": "Product Quantization (PQ) compresses high-dimensional vectors by splitting them into sub-vectors and quantizing each independently. Combined with IVF (IVF+PQ), this enables billion-scale retrieval."},
    {"id": "doc_3", "text": "BM25 is a probabilistic retrieval model that scores documents based on term frequency and inverse document frequency. It is the gold standard for keyword search and a core component of hybrid RAG."},
    {"id": "doc_4", "text": "Reciprocal Rank Fusion (RRF) merges ranked lists from multiple retrieval systems without requiring calibrated scores. The formula: score = 1/(k + rank) where k=60 is the standard default."},
    {"id": "doc_5", "text": "Dense retrieval encodes queries and documents into embedding vectors and retrieves by cosine similarity. Models like BGE-M3, E5-large, and text-embedding-3-small are commonly used."},
    {"id": "doc_6", "text": "Hybrid search combines BM25 and dense retrieval. BM25 excels at exact keyword matches (SKU-4821, model names, IDs). Dense retrieval excels at semantic paraphrases ('car' matches 'automobile')."},
    {"id": "doc_7", "text": "Pinecone is a managed vector database optimized for production RAG. It supports hybrid search with a sparse-dense index, namespace isolation for multi-tenancy, and serverless billing."},
    {"id": "doc_8", "text": "Weaviate is an open-source vector database with built-in hybrid search (BM25 + dense, alpha-weighted). It supports GraphQL queries and multi-tenancy via class isolation."},
    {"id": "doc_9", "text": "Qdrant is a self-hosted vector database written in Rust. It supports filtering by metadata at the ANN search level (not post-filter), preserving recall under selective filters."},
    {"id": "doc_10", "text": "ColBERT uses late interaction: both query and document are encoded to per-token vectors, then scored by MaxSim (sum of max cosine similarities across token pairs). Better than bi-encoders on specialized domains."},
    {"id": "doc_11", "text": "Reranking is a two-stage retrieval strategy. Stage 1: fast approximate retrieval (BM25 or dense) to get top-k candidates. Stage 2: cross-encoder reranker scores each candidate against the query more precisely."},
]

print(f"Corpus: {len(CORPUS)} documents")

## 2. BM25 Index

In [ ]:
from rank_bm25 import BM25Okapi
import re

def tokenize(text: str) -> list[str]:
    """Simple whitespace + lowercase tokenization."""
    return re.sub(r"[^a-zA-Z0-9\-]", " ", text.lower()).split()

# Tokenize corpus
tokenized_corpus = [tokenize(doc["text"]) for doc in CORPUS]
bm25 = BM25Okapi(tokenized_corpus)

def bm25_retrieve(query: str, k: int = 5) -> list[dict]:
    """Return top-k documents by BM25 score."""
    scores = bm25.get_scores(tokenize(query))
    ranked_indices = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)[:k]
    return [
        {"id": CORPUS[i]["id"], "text": CORPUS[i]["text"], "bm25_score": float(scores[i]), "rank": rank + 1}
        for rank, i in enumerate(ranked_indices)
    ]

# Test
results = bm25_retrieve("What is BM25?")
for r in results[:3]:
    print(f"[{r['rank']}] {r['id']} (score: {r['bm25_score']:.3f})")
    print(f"    {r['text'][:80]}...\n")

## 3. Dense Index (sentence-transformers + FAISS)

In [ ]:
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer

# Load a small but capable embedding model
embed_model = SentenceTransformer("BAAI/bge-small-en-v1.5")
print(f"Embedding dimension: {embed_model.get_sentence_embedding_dimension()}")

# Encode corpus
corpus_texts = [doc["text"] for doc in CORPUS]
corpus_embeddings = embed_model.encode(
    corpus_texts,
    normalize_embeddings=True,  # normalize for cosine similarity via inner product
    show_progress_bar=True
)
print(f"Embeddings shape: {corpus_embeddings.shape}")

# Build FAISS index (IndexFlatIP = exact inner product on normalized vectors = cosine)
dim = corpus_embeddings.shape[1]
faiss_index = faiss.IndexFlatIP(dim)
faiss_index.add(corpus_embeddings.astype(np.float32))
print(f"FAISS index size: {faiss_index.ntotal} vectors")

In [ ]:
def dense_retrieve(query: str, k: int = 5) -> list[dict]:
    """Return top-k documents by cosine similarity."""
    # BGE models work better with the query prefix
    query_emb = embed_model.encode(
        f"Represent this sentence for searching relevant passages: {query}",
        normalize_embeddings=True
    ).reshape(1, -1).astype(np.float32)

    scores, indices = faiss_index.search(query_emb, k)
    return [
        {
            "id": CORPUS[idx]["id"],
            "text": CORPUS[idx]["text"],
            "dense_score": float(scores[0][rank]),
            "rank": rank + 1
        }
        for rank, idx in enumerate(indices[0])
    ]

# Test
results = dense_retrieve("approximate nearest neighbor graph algorithms")
for r in results[:3]:
    print(f"[{r['rank']}] {r['id']} (score: {r['dense_score']:.3f})")
    print(f"    {r['text'][:80]}...\n")

## 4. Reciprocal Rank Fusion (RRF)

In [ ]:
def rrf_merge(ranked_lists: list[list[dict]], k: int = 60, top_n: int = 5) -> list[dict]:
    """
    Merge multiple ranked lists using Reciprocal Rank Fusion.
    
    ranked_lists: list of ranked result lists, each containing dicts with 'id' and 'text'
    k: RRF hyperparameter (default 60 is robust across tasks)
    top_n: number of final results to return
    """
    scores = {}
    doc_texts = {}  # preserve text for final output

    for ranked_list in ranked_lists:
        for rank, doc in enumerate(ranked_list, start=1):
            doc_id = doc["id"]
            if doc_id not in scores:
                scores[doc_id] = 0.0
                doc_texts[doc_id] = doc["text"]
            scores[doc_id] += 1.0 / (k + rank)

    # Sort by RRF score
    sorted_docs = sorted(scores.items(), key=lambda x: x[1], reverse=True)[:top_n]
    return [
        {"id": doc_id, "text": doc_texts[doc_id], "rrf_score": rrf_score, "rank": rank + 1}
        for rank, (doc_id, rrf_score) in enumerate(sorted_docs)
    ]


def hybrid_retrieve(query: str, k: int = 5, bm25_k: int = 10, dense_k: int = 10) -> list[dict]:
    """Full hybrid retrieval: BM25 + dense → RRF merge."""
    bm25_results  = bm25_retrieve(query, k=bm25_k)
    dense_results = dense_retrieve(query, k=dense_k)
    return rrf_merge([bm25_results, dense_results], top_n=k)


# Test
results = hybrid_retrieve("What is RRF and how does it merge results?")
for r in results:
    print(f"[{r['rank']}] {r['id']} (rrf: {r['rrf_score']:.4f})")
    print(f"    {r['text'][:80]}...\n")

## 5. Generate with the configured LLM provider

In [ ]:
from ai_client import generate

def generate_answer(query: str, context_docs: list[dict]) -> str:
    context = "\n\n".join(
        f"[{doc['id']}] {doc['text']}" for doc in context_docs
    )
    return generate(
        prompt=f"Context:\n{context}\n\nQuestion: {query}",
        system="Answer the question using only the provided context. Cite the document ID for each claim.",
        max_tokens=512,
    )


def hybrid_rag(query: str) -> str:
    docs = hybrid_retrieve(query, k=5)
    return generate_answer(query, docs)


# End-to-end test
query = "When should I use IVF+PQ instead of HNSW?"
print(f"Query: {query}\n")
print(hybrid_rag(query))

## 6. Comparison: BM25-only vs. Dense-only vs. Hybrid

Run 4 representative queries and compare the top-3 results per strategy.

In [ ]:
TEST_QUERIES = [
    # Exact keyword — BM25 should excel
    "IVF+PQ indexing",
    # Semantic paraphrase — dense should excel
    "How do I reduce memory usage for billions of vectors?",
    # Mixed — hybrid wins
    "HNSW versus IVF for approximate nearest neighbors",
    # Product code style — BM25 critical
    "BM25 term frequency scoring",
]

for query in TEST_QUERIES:
    print(f"\n{'='*60}")
    print(f"QUERY: {query}")
    print(f"{'='*60}")

    for strategy, fn in [("BM25", bm25_retrieve), ("Dense", dense_retrieve), ("Hybrid", hybrid_retrieve)]:
        results = fn(query, k=3)
        print(f"\n  [{strategy}]")
        for r in results:
            print(f"    {r['rank']}. {r['id']}: {r['text'][:60]}...")

## 7. Tuning the RRF k Parameter

The k parameter in RRF controls how much weight the top results get relative to lower-ranked ones. Lower k → top results dominate. Higher k → more uniform weighting.

In [ ]:
query = "approximate nearest neighbor search algorithms"

bm25_r  = bm25_retrieve(query, k=10)
dense_r = dense_retrieve(query, k=10)

print(f"Query: {query}\n")
for k_val in [1, 10, 60, 120]:
    merged = rrf_merge([bm25_r, dense_r], k=k_val, top_n=3)
    ids = [r["id"] for r in merged]
    print(f"  k={k_val:3d}: {ids}")
print("\nNote: k=60 is the standard default (robust across tasks).")

## 8. Key Takeaways

| Retriever | Wins When | Loses When |
|---|---|---|
| **BM25** | Query contains exact keywords, codes, product IDs | Query is a semantic paraphrase ("automobile" doesn't match "car") |
| **Dense** | Query is semantic, paraphrased, or conceptual | Query contains rare domain tokens not seen in training |
| **Hybrid (RRF)** | Default choice — gets both | Small corpora (<1K docs): overhead not worth it |

**RRF vs. weighted sum:** RRF doesn't require calibrated scores — just ranks. This makes it score-scale agnostic (BM25 scores are unbounded; cosine similarity is -1 to 1). For production, RRF is almost always preferred over `alpha * bm25_score + (1-alpha) * dense_score`.